# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadM434-prog/Flyrank-ML-Internship-Assignment1/blob/main/work/notebooks/w01_research_question.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

### Lane: Refresh / Content Opportunity Ranking

I chose this lane because the starter dataset is built around content refresh decisions — each row is one page with 90-day search and engagement metrics plus freshness fields. An editor cannot manually review 30,000 pages across 32 clients, but the data shows real prioritization pressure: more than half of pages are on a declining trend, and roughly a third have not been updated in over 90 days. This lane turns that messy inventory into a ranked review queue — which pages to refresh, protect, prune, or monitor first — which is a concrete decision with a clear customer (the content/editor team) and a measurable wrong-call cost (wasted review time vs. missed declines on high-traffic pages).

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**Decision:** Which pages should the team review first — for refresh, expansion, protection, pruning, or monitoring?

With 30,000 pages in the starter slice alone, nobody can inspect everything. The work should produce a **ranked review queue** so limited editor time goes to pages where action is most likely to matter.

**Who acts, and what they do:** A content editor or SEO strategist uses the queue to pick pages to open, diagnose, and act on — update stale copy, rewrite a weak title/meta, consolidate thin pages, protect a still-strong asset, or deprioritize low-demand pages. They are not blindly refreshing every flagged page; the score and reason codes tell them *why* a page surfaced.

**Cost of a wrong call:**

- **Too high on the queue (false urgency):** Editor hours spent on a low-impact page — e.g. a thin page with few impressions that was declining only in noise. Cost = wasted review and refresh effort.
- **Too low or missed (false calm):** A visible, high-impression page that is declining or going stale stays off the radar. Cost = lost clicks and engagement that a timely refresh might have recovered — we cannot prove causally, but the opportunity window closes.
- **Wrong action type:** Flagging "refresh" when the page needs "protect" or "prune." Cost = misfired edits that do not match the page's actual situation.

Because errors cut both ways, the method needs to be **transparent** (reason codes, not a black-box score), **volume-aware** (do not panic over 12 impressions), and **honest about uncertainty** — decision support, not a guarantee that refreshing will pay off.

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [8]:
from pathlib import Path
import pandas as pd
import os

# Define the repository name and URL
repo_name = "Flyrank-ML-Internship-Assignment1"
repo_url = f"https://github.com/MuhammadM434-prog/{repo_name}"

# Define the expected path for the cloned repository in Colab
colab_repo_path = Path(f"/content/{repo_name}")

# Check if the repository is already cloned at the expected Colab location
if not colab_repo_path.exists():
    print(f"Cloning {repo_url} into /content/...")
    # Change directory to /content/ first to ensure cloning happens there
    current_cwd = Path.cwd()
    os.chdir("/content/")
    !git clone {repo_url}
    os.chdir(current_cwd) # Change back to original cwd
    print(f"Repository {repo_name} cloned.")
else:
    print(f"Repository {repo_name} already exists at {colab_repo_path.resolve()}.")

# --- Root finding logic ---
root = None

# 1. Check if the cloned repo path itself is the root
if (colab_repo_path / "data/raw/content_refresh_anonymized.csv").exists():
    root = colab_repo_path.resolve()
else:
    # 2. Otherwise, use the original logic to search from current working directory and its parents
    # This covers cases where the notebook is run locally from within the repo structure
    # or if the repo was cloned elsewhere but is still accessible in the path hierarchy.
    current_dir = Path.cwd().resolve()
    for candidate in [current_dir, *current_dir.parents]:
        if (candidate / "data/raw/content_refresh_anonymized.csv").exists():
            root = candidate
            break

if root is None:
    raise FileNotFoundError(
        "Starter CSV not found. Please ensure the repository is cloned and "
        "the data file 'data/raw/content_refresh_anonymized.csv' exists "
        "relative to the expected repository root, or that the notebook is "
        "run from a location where the data file is accessible."
    )

data_path = root / "data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(data_path)

n = len(df)
declining = df["trend_direction"] == "down"
stale = df["days_since_last_update"] > 90
high_volume = df["impressions_90d"] >= 1_000
declining_high_volume = declining & high_volume

print(f"Starter dataset: {data_path.relative_to(root)}")
print(f"Rows: {n:,}  |  Clients: {df['client_id'].nunique()}")
print()
print(f"1) Declining trend (trend_direction == 'down'): {declining.sum():,} pages ({100 * declining.mean():.2f}%) ")
print(f"2) Stale content (days_since_last_update > 90): {stale.sum():,} pages ({100 * stale.mean():.2f}%) ")
print(
    f"3) Declining AND high-volume (>= 1,000 impressions in 90d): "
    f"{declining_high_volume.sum():,} pages ({100 * declining_high_volume.mean():.2f}%) ")


Repository Flyrank-ML-Internship-Assignment1 already exists at /content/Flyrank-ML-Internship-Assignment1.
Starter dataset: data/raw/content_refresh_anonymized.csv
Rows: 30,000  |  Clients: 32

1) Declining trend (trend_direction == 'down'): 16,262 pages (54.21%) 
2) Stale content (days_since_last_update > 90): 9,345 pages (31.15%) 
3) Declining AND high-volume (>= 1,000 impressions in 90d): 8,031 pages (26.77%) 


**What these numbers show (and why this lane is worth 7 weeks):**

1. **54% declining** — More than half of pages are on a downward trend. An editor cannot treat every page as urgent; something has to rank them.
2. **31% stale** — Nearly one in three pages has not been updated in over 90 days, a direct refresh signal sitting next to performance metrics.
3. **8,031 high-stakes pages** — About 27% of the inventory is both declining *and* has at least 1,000 impressions in 90 days. These are the pages where a wrong queue position wastes real visibility — exactly the subset a content opportunity ranking system should prioritize first.

## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

### What this work **can** say

- **Observed:** In this anonymized starter slice (30,000 pages, 32 clients, trailing 90-day metrics), we can describe measured patterns — how many pages are declining, stale, or high-volume, and how those groups differ on clicks, impressions, and freshness fields.
- **Directional:** We can report **associations** in the data — e.g. pages with longer `days_since_last_update` or lower recent click totals **tend to appear** alongside declining trend labels. Wording stays "associated with" / "co-occurs with," not "causes."
- **Decision-support:** We can produce a **ranked review queue** with reason codes (stale + visible, declining with volume, thin page with demand, etc.) so editors spend limited time on pages that **look worth opening first** in this portfolio. Validation can report ranking quality (e.g. precision@K on held-out clients) — how well the queue surfaces pages that match our label definition, not a guarantee of business outcome.
- **Measured comparisons:** Any group difference we report will include counts, base rates, and filters (e.g. "among pages with ≥1,000 impressions…") so small buckets do not become dramatic headlines.

### What this work **never** will say

- **Causal proof:** We will not claim that refreshing, rewriting, or pruning a page **will increase** rankings, clicks, or revenue. This is cross-sectional observational data — no randomized intervention, no matched refresh experiment.
- **"Predicting Google":** We are not modeling Google's ranking algorithm. We are scoring pages in one anonymized content portfolio using search and analytics metrics already collected.
- **Guaranteed payoff:** A "declining" or "high score" flag is a **review trigger**, not proof that action will pay off. Many declines may be noise, seasonality, or competition — the queue prioritizes inspection, not certainty.
- **Universal rules:** Findings from this slice do not automatically transfer to every client, content type, or future time window without re-validation on held-out groups and (later) the warehouse release.

These boundaries keep the lane honest: **help editors decide where to look first**, not promise what happens after they edit.

## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.